# Обучение RuBERT для классификации опасного контента

Классы:
- **0** — Насилие
- **1** — Ненависть
- **2** — Суицид
- **3** — Дезинформация
- **4** — Нейтральный

## Шаги:
1. Загрузить `dataset_for_training.csv` в Google Drive
2. Запустить все ячейки по порядку
3. Обученная модель сохранится в Google Drive

In [ ]:
!pip install transformers datasets accelerate torch scikit-learn pandas openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Загрузка датасета

Загрузите `dataset_for_training.csv` в Google Drive перед запуском.

Файл создаётся скриптом `prepare_datasets.py` из 6 датасетов:
- Ручной датасет (5K)
- Depressive data VK (64K)
- EUvsDisinfo дезинформация (9K)
- Inappropriate messages (125K)
- Toxic comments 2ch/Pikabu (14K)
- Sensitive topics 18 тем (33K)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================
# УКАЖИТЕ ПУТЬ К ВАШЕМУ ФАЙЛУ
# ============================================
CSV_FILE_PATH = '/content/drive/MyDrive/dataset_for_training.csv'
# ============================================

df = pd.read_csv(CSV_FILE_PATH, encoding='utf-8')

label_names = {
    0: 'Насилие',
    1: 'Ненависть',
    2: 'Суицид',
    3: 'Дезинформация',
    4: 'Нейтральный'
}

print(f"Загружено примеров: {len(df):,}")
print(f"Колонки: {list(df.columns)}")
print(f"\nРаспределение по классам:")
for label in sorted(df['label'].unique()):
    count = (df['label'] == label).sum()
    pct = count / len(df) * 100
    print(f"  [{label}] {label_names[label]:15s}: {count:6,} ({pct:5.1f}%)")

print(f"\nПримеры:")
for label in sorted(df['label'].unique()):
    sample = df[df['label'] == label].iloc[0]
    print(f"  [{label_names[label]}] {str(sample['text'])[:80]}...")

In [ ]:
# Визуализация распределения
plt.figure(figsize=(10, 5))
counts = df['label'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#9b59b6', '#3498db', '#2ecc71']
bars = plt.bar(range(5), [counts.get(i, 0) for i in range(5)], color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Класс', fontsize=13)
plt.ylabel('Количество', fontsize=13)
plt.title(f'Распределение классов ({len(df):,} примеров)', fontsize=15)
plt.xticks(range(5), [label_names[i] for i in range(5)], fontsize=11)
plt.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height):,}',
             ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Токенизация и подготовка данных

In [ ]:
# Разделение на train/val/test (70/15/15)
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"\nTrain распределение:")
for label in sorted(train_df['label'].unique()):
    print(f"  [{label}] {label_names[label]:15s}: {(train_df['label']==label).sum():,}")

# Загрузка токенизатора
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Преобразование в Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

# Токенизация
print("\nТокенизация...")
train_dataset = train_dataset.map(tokenize_function, batched=True, desc="Train")
val_dataset = val_dataset.map(tokenize_function, batched=True, desc="Val")
test_dataset = test_dataset.map(tokenize_function, batched=True, desc="Test")

# Формат PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✅ Токенизация завершена!")

## 3. Загрузка модели

In [ ]:
num_labels = 5

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="single_label_classification"
)

model.to(device)
print(f"✅ Модель загружена на {device}")
print(f"   Параметров: {sum(p.numel() for p in model.parameters()):,}")

## 4. Настройка обучения

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir='./logs',
    logging_steps=50,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

print("✅ Параметры обучения:")
print(f"   Эпох: {training_args.num_train_epochs}")
print(f"   Batch: {training_args.per_device_train_batch_size}")
print(f"   LR: {training_args.learning_rate}")
print(f"   FP16: {training_args.fp16}")
print(f"   Warmup: {training_args.warmup_ratio}")

## 5. Обучение

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Начало обучения...")
print(f"   Train: {len(train_dataset):,} примеров")
print(f"   Val:   {len(val_dataset):,} примеров")
print(f"   Шагов на эпоху: {len(train_dataset) // training_args.per_device_train_batch_size}")
print()

trainer.train()

print("\n✅ Обучение завершено!")

## 6. Оценка на тестовой выборке

In [ ]:
# Предсказания
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Общая точность
test_accuracy = accuracy_score(true_labels, pred_labels)
test_f1 = f1_score(true_labels, pred_labels, average='macro')
print(f"\n🎯 Точность: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 F1 Macro: {test_f1:.4f}")

# Детальный отчет
print("\nДетальный отчет:")
print(classification_report(
    true_labels, pred_labels,
    target_names=[label_names[i] for i in range(5)],
    digits=4
))

# Матрица ошибок
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[label_names[i] for i in range(5)],
            yticklabels=[label_names[i] for i in range(5)])
plt.xlabel('Предсказанный класс', fontsize=13)
plt.ylabel('Истинный класс', fontsize=13)
plt.title('Матрица ошибок', fontsize=15)
plt.tight_layout()
plt.show()

# F1 по классам
f1_per_class = f1_score(true_labels, pred_labels, average=None)
plt.figure(figsize=(10, 5))
bars = plt.bar(range(5), f1_per_class, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Класс', fontsize=13)
plt.ylabel('F1-Score', fontsize=13)
plt.title('F1-Score по классам', fontsize=15)
plt.xticks(range(5), [label_names[i] for i in range(5)])
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, f1_per_class):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Сохранение модели

In [ ]:
# Сохранение в Google Drive
model_save_path = '/content/drive/MyDrive/rubert_dangerous_content_v4'

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"✅ Модель сохранена: {model_save_path}")
print(f"\nФайлы модели:")
import os
for f in os.listdir(model_save_path):
    size = os.path.getsize(os.path.join(model_save_path, f)) / (1024*1024)
    print(f"   {f} ({size:.1f} MB)")

print(f"\n📦 Для использования в GUI:")
print(f"   Скачайте папку {model_save_path}")
print(f"   Укажите её в content_analyzer_gui.py")

## 8. Быстрый тест модели

In [ ]:
def predict_text(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)[0]
    pred = torch.argmax(probs).item()
    return label_names[pred], {label_names[i]: f"{probs[i].item():.3f}" for i in range(5)}

test_texts = [
    "Убью тебя, если не отдашь деньги!",
    "Все мигранты — преступники и воры",
    "Не хочу больше жить, всё бессмысленно",
    "Вакцины содержат наночипы для контроля людей",
    "Сегодня хорошая погода, пойдём гулять",
    "Разобью тебе лицо при встрече",
    "Геи — извращенцы, их нужно лечить",
    "Прощайте все, ухожу навсегда из этого мира",
    "Земля плоская, NASA врёт всем",
    "Купил новый телефон, доволен покупкой",
]

print("🧪 Тестирование модели:\n")
for text in test_texts:
    label, probs = predict_text(text)
    print(f"  [{label:15s}] {text[:60]}")
    print(f"                   {probs}")
    print()